## Libraries

In [ ]:
# pip install rapidfuzz

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------- ----- 1.3/1.5 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 6.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rapidfuzz import process

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler, OneHotEncoder



from sklearn.linear_model import Lasso, Ridge


## Import Dataset

In [ ]:
#Import the training dataset
train = pd.read_csv('project_data/train.csv')
train.head()

,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
3,29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
4,10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0


In [ ]:
#Import the test dataset
test = pd.read_csv('project_data/test.csv')

## New Features

In [ ]:

test['mileage_per_year'] = test['mileage'] / (2025 - test['year'])
train['mileage_per_year'] = train['mileage'] / (2025 - train['year'])

In [485]:
test['tax_engineSize'] = test['tax'] *  test['engineSize']
train['tax_engineSize'] = train['tax'] * train['engineSize']

In [486]:
test['age_mileage'] = (2025 - test['year']) *  test['mileage']
train['age_mileage'] = (2025 - train['year']) * train['mileage']

## Inconsistent Data

#### Negative Data

In [ ]:
#Features where exists negavite data 
negative_features = ['mileage', 'mpg', 'engineSize', 'previousOwners', 'tax', 'mileage_per_year', 'tax_engineSize', 'age_mileage']

In [ ]:
#Get the absolute value
train[negative_features] = train[negative_features].abs()
test[negative_features] = test[negative_features].abs()

In [489]:
train[negative_features].min()

mileage             1.000000
mpg                 1.100000
engineSize          0.000000
previousOwners      0.000000
tax                 0.000000
mileage_per_year    0.041667
tax_engineSize      0.000000
age_mileage         5.000000
dtype: float64

#### Standardize

In [ ]:
#Lists with the correct values for the features
correct_transmission = ['Manual', 'Semi-Auto', 'Automatic', 'Unknown']
correct_fuelType = ['Petrol', 'Diesel', 'Hybrid', 'Other', 'Unknown']

#Apply rapidfuzz to the features
def clean_strings(x, l):
    if pd.isna(x):
        return 'Unknown'
    x = x.strip().title()
    match, score, _ = process.extractOne(x, l)
    
    if score > 80:
        return match
    else:
        return 'Unknown'

##### Transmission

In [31]:
train['transmission'] = train['transmission'].apply(lambda x: clean_strings(x, correct_transmission))

In [18]:
test['transmission'] = test['transmission'].apply(lambda x: clean_strings(x, correct_transmission))

In [32]:
train['transmission'].value_counts()

transmission
Manual       40798
Semi-Auto    16867
Automatic    15205
Unknown       3103
Name: count, dtype: int64

##### FuelType

In [35]:
for df in [train, test]:
    df['fuelType'] = df['fuelType'].apply(lambda x: clean_strings(x, correct_fuelType))

In [36]:
train['fuelType'].value_counts()

fuelType
Petrol     40353
Diesel     30255
Unknown     3020
Hybrid      2184
Other        161
Name: count, dtype: int64

## Training and Test

In [ ]:
#Drop the target from the training dataset and create a new dataset only with the target
X_trainval = train.drop(columns=['price', 'carID'])
y_trainval = train['price']

## Dealing with missing values

In [ ]:
#Get the median values of the numeric features in the trainig dataset
median_values = X_trainval.median(numeric_only=True)

#Apply those median values into the training dataset and test dataset
for feature in median_values.index:
    X_trainval[feature].fillna(median_values[feature], inplace=True)

for feature in median_values.index:
    test[feature].fillna(median_values[feature], inplace=True)
        

C:\Users\User\AppData\Local\Temp\ipykernel_2996\220773069.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_trainval[feature].fillna(median_values[feature], inplace=True)
C:\Users\User\AppData\Local\Temp\ipykernel_2996\220773069.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy

In [ ]:
#Fill the NA's with Unknown for the categorical features
for feature in ['Brand', 'model', 'transmission', 'fuelType']:
    X_trainval[feature].fillna('Unknown', inplace=True)
    test[feature].fillna('Unknown', inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_2996\2865857449.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_trainval[feature].fillna('Unknown', inplace=True)
C:\Users\User\AppData\Local\Temp\ipykernel_2996\2865857449.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

In [493]:
X_trainval.isna().sum()

Brand               0
model               0
year                0
transmission        0
mileage             0
fuelType            0
tax                 0
mpg                 0
engineSize          0
paintQuality%       0
previousOwners      0
hasDamage           0
mileage_per_year    0
tax_engineSize      0
age_mileage         0
dtype: int64

In [494]:
test.isna().sum()

carID               0
Brand               0
model               0
year                0
transmission        0
mileage             0
fuelType            0
tax                 0
mpg                 0
engineSize          0
paintQuality%       0
previousOwners      0
hasDamage           0
mileage_per_year    0
tax_engineSize      0
age_mileage         0
dtype: int64

## Data Types

In [ ]:
#List with features to change variable type (float -> int)
numeric_cols = ['year', 'paintQuality%', 'previousOwners']

for feature in numeric_cols:
    X_trainval[feature] = X_trainval[feature].astype(int)

for feature in numeric_cols:
    test[feature] = test[feature].astype(int)

In [496]:
X_trainval.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75973 entries, 0 to 75972
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Brand             75973 non-null  object 
 1   model             75973 non-null  object 
 2   year              75973 non-null  int64  
 3   transmission      75973 non-null  object 
 4   mileage           75973 non-null  float64
 5   fuelType          75973 non-null  object 
 6   tax               75973 non-null  float64
 7   mpg               75973 non-null  float64
 8   engineSize        75973 non-null  float64
 9   paintQuality%     75973 non-null  int64  
 10  previousOwners    75973 non-null  int64  
 11  hasDamage         75973 non-null  float64
 12  mileage_per_year  75973 non-null  float64
 13  tax_engineSize    75973 non-null  float64
 14  age_mileage       75973 non-null  float64
dtypes: float64(8), int64(3), object(4)
memory usage: 8.7+ MB


## Separate the categorical and numerical data

In [497]:
num_cols = X_trainval.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_trainval.select_dtypes(include=['object', 'bool']).columns

In [498]:
X_trainval_num = X_trainval[num_cols]
X_trainval_cat = X_trainval[cat_cols]

In [499]:
X_test_num = test[num_cols]
X_test_cat = test[cat_cols]

## Normalizing the data

#### Numerical

In [ ]:
# Using MinMaxScaler to normalize the numerical features
scaler = MinMaxScaler()

In [501]:
X_trainval_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_trainval_num),
    columns=num_cols
)

X_trainval_num_scaled

,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage,mileage_per_year,tax_engineSize,age_mileage
0,0.851852,0.087988,0.250000,0.021966,0.303030,0.500000,0.666667,0.0,0.018668,0.058398,0.046582
1,0.907407,0.014204,0.250000,0.099638,0.227273,0.395161,0.166667,0.0,0.004521,0.060484,0.005013
2,0.907407,0.011217,0.250000,0.084735,0.227273,0.443548,0.666667,0.0,0.003570,0.060484,0.003959
3,0.888889,0.028177,0.250000,0.137535,0.151515,0.395161,0.333333,0.0,0.007686,0.040323,0.011602
4,0.907407,0.003093,0.250000,0.088780,0.227273,0.774194,0.500000,0.0,0.000985,0.060484,0.001092
...,...,...,...,...,...,...,...,...,...,...,...
75968,0.833333,0.044827,0.215517,0.111135,0.303030,0.620968,0.000000,0.0,0.008560,0.069522,0.026370
75969,0.796296,0.161403,0.344828,0.099638,0.303030,0.298387,0.333333,0.0,0.025682,0.111235,0.113933
75970,0.870370,0.034994,0.250000,0.140302,0.151515,0.451613,0.500000,0.0,0.008353,0.040323,0.016468
75971,0.833333,0.213843,0.215517,0.125612,0.303030,0.588710,0.333333,0.0,0.040832,0.069522,0.125790


In [502]:
X_test_num_scaled = pd.DataFrame(
    scaler.transform(X_test_num),
    columns=num_cols
)

#### Categorical

In [ ]:
#Using OneHot Enconder to normalize the categorical features
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

In [504]:
X_trainval_cat_encoded = pd.DataFrame(
    encoder.fit_transform(X_trainval_cat),
    columns=encoder.get_feature_names_out(cat_cols)
)


In [505]:
X_test_cat_encoded = pd.DataFrame(
    encoder.transform(X_test_cat),
    columns=encoder.get_feature_names_out(cat_cols)
)

c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [506]:
X_trainval_full = pd.concat([X_trainval_num_scaled, X_trainval_cat_encoded], axis=1)
X_test_full = pd.concat([X_test_num_scaled, X_test_cat_encoded], axis=1)

## Feature Selection

In [ ]:
#Using Lasso to select the features
lasso = Lasso(alpha=0.1, random_state=42)

In [508]:
lasso.fit(X_trainval_full, y_trainval)

,alpha,0.1
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,42
,selection,'cyclic'


In [ ]:
#Only keeping the features that don't shrink to zero 
lasso_features_to_select = X_trainval_full.columns[(lasso.coef_ != 0)]
print(f"Lasso kept {len(lasso_features_to_select)} features")

Lasso kept 572 features


In [ ]:
#Applying those features to the final dataset
X_trainval_final = X_trainval_full[lasso_features_to_select]
X_test_final = X_test_full[lasso_features_to_select]

In [511]:
X_trainval_final.head()

,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,mileage_per_year,tax_engineSize,age_mileage,...,fuelType_diese,fuelType_diesel,fuelType_etro,fuelType_hybrid,fuelType_iese,fuelType_iesel,fuelType_petro,fuelType_petrol,fuelType_ther,fuelType_ybrid
0,0.851852,0.087988,0.25,0.021966,0.303030,0.500000,0.666667,0.018668,0.058398,0.046582,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.907407,0.014204,0.25,0.099638,0.227273,0.395161,0.166667,0.004521,0.060484,0.005013,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.907407,0.011217,0.25,0.084735,0.227273,0.443548,0.666667,0.003570,0.060484,0.003959,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.888889,0.028177,0.25,0.137535,0.151515,0.395161,0.333333,0.007686,0.040323,0.011602,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.907407,0.003093,0.25,0.088780,0.227273,0.774194,0.500000,0.000985,0.060484,0.001092,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Export

In [512]:
X_trainval_final.to_csv('X_trainval_preprocessed.csv', index=False)
y_trainval.to_csv('y_trainval.csv', index=False)
X_test_final.to_csv('X_test_preprocessed.csv', index=False)